<a href="https://colab.research.google.com/github/JoshuaFZ/QWEN-0.6B-LORA/blob/main/voicesense_tunning_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SenseVoice 微调 Notebook（colab 版）
加载云盘安装环境

In [ ]:
from google.colab import drive
import os

# 1. 挂载 Google Drive
drive.mount('/content/drive')

# 2. 安装依赖
if 'MODE_WITH_AUTO_TEST' not in os.environ:
    !pip install -U modelscope -q
    !pip install addict soundfile librosa sentencepiece -q
    !pip install modelscope[audio] -f https://modelscope.oss-cn-beijing.aliyuncs.com/releases/repo.html -q

print("\n✅ 云盘已挂载，环境依赖安装完成！")


## 2. 在原有模型基础继续训练

In [ ]:
import os
import shutil
import subprocess
import sys
import torch
from pathlib import Path

print("--- 3. 重新执行微调 (增量训练) ---")
print("Notebook patch version: 2026-05-18-device-v3")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"当前计算设备: {device}")

# 1. 环境与路径准备
if not os.path.exists("./FunASR"):
    !git clone https://github.com/alibaba-damo-academy/FunASR.git

!pip install -U modelscope -q
!pip install addict soundfile librosa sentencepiece -q
!pip install -e ./FunASR -q

abs_cwd = os.getcwd()
abs_funasr_path = os.path.join(abs_cwd, "FunASR")

PROJECT_DIR = Path("/content/drive/MyDrive/SenseVoice_Project")
ORIGINAL_MODEL_PATH = PROJECT_DIR / "SenseVoiceSmall_Original"
# PREV_FINETUNED_PATH = PROJECT_DIR / "sensevoice_finetuned_03"
PREV_FINETUNED_PATH = None
DRIVE_OUTPUT_DIR = PROJECT_DIR / "sensevoice_finetuned_05"
TRAIN_JSONL = Path("/content/drive/MyDrive/branch1/train_with_len_plus_errors_20260519.jsonl")
BASE_MODEL_ID = "iic/SenseVoiceSmall"
local_training_dir = Path(abs_cwd) / "local_model_pkg"

def safe_cleanup(path):
    path = Path(path)
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.is_dir():
        shutil.rmtree(path)

def copy_overlay(src, dst):
    src = Path(src)
    dst = Path(dst)
    if not src.exists():
        return
    for item in src.iterdir():
        target = dst / item.name
        safe_cleanup(target)
        if item.is_dir():
            shutil.copytree(item, target, symlinks=True)
        else:
            shutil.copy2(item, target)

def ensure_base_model():
    from modelscope import snapshot_download

    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    if ORIGINAL_MODEL_PATH.exists() and any(ORIGINAL_MODEL_PATH.iterdir()):
        print("已存在完整基座模型目录:", ORIGINAL_MODEL_PATH)
        return

    print("正在下载完整基座模型:", BASE_MODEL_ID)
    downloaded = Path(snapshot_download(BASE_MODEL_ID, cache_dir="/content/modelscope_cache"))
    safe_cleanup(ORIGINAL_MODEL_PATH)
    shutil.copytree(downloaded, ORIGINAL_MODEL_PATH, symlinks=True)
    print("基座模型已保存到 Google Drive:", ORIGINAL_MODEL_PATH)

if not TRAIN_JSONL.exists():
    raise FileNotFoundError(f"训练数据不存在: {TRAIN_JSONL}")

# 2. 下载/复用完整基座模型。不要用 AutoModel 预热缓存，避免触发 model 模块远程代码导入 warning。
ensure_base_model()

# 3. 构造完整本地模型包：先复制基座模型，再用上次微调输出覆盖。
safe_cleanup(local_training_dir)
shutil.copytree(ORIGINAL_MODEL_PATH, local_training_dir, symlinks=True)

# if PREV_FINETUNED_PATH.exists() and any(PREV_FINETUNED_PATH.iterdir()):
#     print("发现上次微调模型，覆盖到本地训练包:", PREV_FINETUNED_PATH)
#     copy_overlay(PREV_FINETUNED_PATH, local_training_dir)
# else:
#     print("未发现上次微调模型，本次从完整基座模型开始训练。")

if PREV_FINETUNED_PATH and PREV_FINETUNED_PATH.exists() and any(PREV_FINETUNED_PATH.iterdir()):
    print("发现上次微调模型，覆盖到本地训练包:", PREV_FINETUNED_PATH)
    copy_overlay(PREV_FINETUNED_PATH, local_training_dir)
else:
    print("未使用上次微调模型，本次从完整基座模型开始训练。")


# 旧输出目录里的 config.yaml 可能带有 train_conf.device。
# FunASR train.py 会把顶层 device 单独传给 Trainer；train_conf.device 残留会导致 device 被传两次。
config_yaml_path = local_training_dir / "config.yaml"
if config_yaml_path.exists():
    try:
        import yaml

        config_data = yaml.safe_load(config_yaml_path.read_text(encoding="utf-8")) or {}
        train_conf = config_data.get("train_conf")
        if isinstance(train_conf, dict) and "device" in train_conf:
            removed_device = train_conf.pop("device")
            config_yaml_path.write_text(
                yaml.safe_dump(config_data, allow_unicode=True, sort_keys=False),
                encoding="utf-8",
            )
            print("已从 config.yaml 删除 train_conf.device:", removed_device)
    except Exception as exc:
        print("检查/清理 config.yaml 中 train_conf.device 失败:", exc)

# 让 trust_remote_code 能找到本地模型仓库里的 model.py 等远程代码文件。
if str(local_training_dir) not in sys.path:
    sys.path.insert(0, str(local_training_dir))

print("本地训练模型包:", local_training_dir)
print("包含文件:", sorted(p.name for p in local_training_dir.iterdir())[:30])

safe_cleanup(DRIVE_OUTPUT_DIR)

print(f"🚀 开始增量强化微调 (设备: {device})...")

env = os.environ.copy()
env["PYTHONPATH"] = f"{local_training_dir}:{abs_cwd}:{env.get('PYTHONPATH', '')}"
env["CUDA_VISIBLE_DEVICES"] = "0"
env["HYDRA_FULL_ERROR"] = "1"
env["PYTHONUNBUFFERED"] = "1"
TRAIN_LOG_PATH = PROJECT_DIR / "sensevoice_finetuned_resume_train.log"

train_cmd = [
    "torchrun",
    "--nproc_per_node=1",
    "FunASR/funasr/bin/train.py",
    "++model=" + str(local_training_dir),
    "++trust_remote_code=True",
    "++train_data_set_list=" + str(TRAIN_JSONL),
    "++valid_data_set_list=" + str(TRAIN_JSONL),
    "++dataset_conf.batch_type=token",
    "++dataset_conf.batch_size=2000",
    "++dataset_conf.data_split_num=1",
    "++dataset_conf.num_workers=0",
    "++dataset_conf.filter_conf.max_length=100000",
    "++dataset_conf.filter_conf.min_length=0",
    "++dataset_conf.filter_conf.token_max_length=100000",
    "++dataset_conf.filter_conf.token_min_length=0",
    "++train_conf.optim_conf.lr=0.00002",
    "++train_conf.max_epoch=5",
    "++device=" + str(device),
    "++output_dir=" + str(DRIVE_OUTPUT_DIR),
]

print("执行命令:")
print(" ".join(train_cmd))
if any("{local_training_dir}" in arg for arg in train_cmd):
    raise RuntimeError("训练命令仍包含未展开的 {local_training_dir}，请重新打开更新后的 notebook。")
if any("train_conf.device" in arg for arg in train_cmd):
    raise RuntimeError("训练命令仍包含 ++train_conf.device，请重新打开 2026-05-18-device-v3 版本 notebook。")

print("训练日志:", TRAIN_LOG_PATH)
with open(TRAIN_LOG_PATH, "w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        train_cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)
    return_code = process.wait()

if return_code != 0:
    print("\n❌ 训练失败，下面是日志最后 200 行:")
    try:
        tail_lines = TRAIN_LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()[-200:]
        print("\n".join(tail_lines))
    except Exception as exc:
        print("读取训练日志失败:", exc)
    raise subprocess.CalledProcessError(return_code, train_cmd)

print(f"\n✅ 任务执行完毕！输出在 {DRIVE_OUTPUT_DIR}")


## 3.加载模型并测试

In [ ]:
import datetime
import html
import json
import os
import re
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from tqdm import tqdm

try:
    from funasr import AutoModel
except ModuleNotFoundError:
    print('未找到 funasr，正在安装测试依赖...')
    import subprocess
    import sys

    funasr_repo = Path('FunASR').resolve()
    if not funasr_repo.exists():
        subprocess.run(['git', 'clone', 'https://github.com/alibaba-damo-academy/FunASR.git'], check=True)

    subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'modelscope', '-q'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'addict', 'soundfile', 'librosa', 'sentencepiece', '-q'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(funasr_repo), '-q'], check=True)

    if str(funasr_repo) not in sys.path:
        sys.path.insert(0, str(funasr_repo))
    from funasr import AutoModel


# --- 配置区 ---
DATA_ROOT = Path('/content/drive/MyDrive/branch1')
TEST_DIRS = [
    DATA_ROOT / 'asr_benchmark',
    DATA_ROOT / 'asr_supplement',
]
SOURCE_JSON_PATH = DATA_ROOT / 'wav_expected.json'
CSV_PATH = DATA_ROOT / 'error_analysis.csv'

PROJECT_DIR = Path('/content/drive/MyDrive/SenseVoice_Project')
RESUME_MODEL_DIR = PROJECT_DIR / 'sensevoice_finetuned_04'
PREV_FINETUNED_DIR = PROJECT_DIR / 'sensevoice_finetuned_03'

# 优先评估最新训练输出；如果还没有 resume 输出，则回退到之前的微调目录。
FINETUNED_MODEL_DIR = RESUME_MODEL_DIR if RESUME_MODEL_DIR.exists() else PREV_FINETUNED_DIR

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
HTML_PATH = Path(f'/content/asr_compare_with_paths_{timestamp}.html')


def clean_text(text):
    """清理 SenseVoice 控制符，保留用于展示的识别文本。"""
    if not text:
        return ''
    return re.sub(r'<\|.*?\|>', '', str(text)).replace(' ', '').strip()


def normalize_text(text):
    """忽略大小写、空格、标点和特殊符号后做自动判定。"""
    if not text:
        return ''
    text = str(text).lower()
    return re.sub(r'[^\w\u4e00-\u9fa5]', '', text)


def relative_audio_path(path):
    path = Path(path)
    try:
        return path.relative_to(DATA_ROOT).as_posix()
    except ValueError:
        return path.as_posix()


def load_expected_mapping(source_json_path):
    expected_mapping = {}
    basename_candidates = {}
    if not source_json_path.exists():
        print(f'⚠️ 未找到标注文件: {source_json_path}')
        return expected_mapping

    try:
        raw_data = json.loads(source_json_path.read_text(encoding='utf-8-sig'))
    except Exception as exc:
        print(f'⚠️ 读取 wav_expected.json 失败: {exc}')
        return expected_mapping

    if isinstance(raw_data, dict) and 'entries' in raw_data:
        data_list = raw_data['entries']
    elif isinstance(raw_data, list):
        data_list = raw_data
    elif isinstance(raw_data, dict):
        data_list = [{'file': key, 'expected': value} for key, value in raw_data.items()]
    else:
        data_list = []

    def normalize_key(value):
        return str(value).strip().replace('\\', '/').lstrip('./')

    def add_expected(key, text):
        key = normalize_key(key)
        if not key:
            return
        expected_mapping[key] = str(text).strip()
        if not key.startswith('asr_benchmark/'):
            expected_mapping[f'asr_benchmark/{key}'] = str(text).strip()
        basename_candidates.setdefault(Path(key).name, set()).add(str(text).strip())

    for item in data_list:
        audio_name = item.get('file') or item.get('wav') or item.get('audio')
        text = item.get('expected') or item.get('text') or item.get('txt') or item.get('label')
        if audio_name and text:
            add_expected(audio_name, text)

    # 只有文件名在标注中不冲突时，才允许用 basename 兜底。
    # clean/noise 等目录下存在同名 wav；直接按 basename 匹配会把 expected 覆盖错。
    for basename, values in basename_candidates.items():
        if len(values) == 1:
            expected_mapping[basename] = next(iter(values))

    return expected_mapping


def generate_text(model, wav_path):
    if model is None:
        return ''
    result = model.generate(input=str(wav_path), disable_pbar=True)
    if not result:
        return ''
    return clean_text(result[0].get('text', ''))


def evaluate_status(expected, base_text, finetuned_text):
    norm_expected = normalize_text(expected)
    if not norm_expected:
        return '无标准答案', '', ''

    base_correct = normalize_text(base_text) == norm_expected
    finetuned_correct = normalize_text(finetuned_text) == norm_expected

    if base_correct and finetuned_correct:
        status = '都对'
    elif base_correct:
        status = '基础对'
    elif finetuned_correct:
        status = '微调对'
    else:
        status = '都不对'

    return status, ' ✅' if base_correct else ' ❌', ' ✅' if finetuned_correct else ' ❌'


def status_html(status):
    if status in {'都对', '微调对'}:
        css_class = 'same'
    elif status in {'基础对', '都不对'}:
        css_class = 'diff'
    else:
        css_class = 'missing'
    return f"<span class='{css_class}'>{html.escape(status)}</span>"


def build_result_row(row, index):
    return f"""
        <tr>
            <td>{index}</td>
            <td><span class="path">{html.escape(row['rel_path'])}</span></td>
            <td><b>{html.escape(row['expected'])}</b></td>
            <td>{html.escape(row['base'])}{row['_base_mark']}</td>
            <td>{html.escape(row['ft'])}{row['_ft_mark']}</td>
            <td>{status_html(row['status'])}</td>
        </tr>
    """


print('--- 1. 加载模型 ---')
print('正在加载原始基础模型...')
base_model = AutoModel(model='iic/SenseVoiceSmall', trust_remote_code=True, disable_update=True)

print('\n正在加载微调后的模型...')
finetuned_model = None
if FINETUNED_MODEL_DIR.exists():
    try:
        finetuned_model = AutoModel(
            model=str(FINETUNED_MODEL_DIR),
            trust_remote_code=True,
            disable_update=True,
        )
        print('✅ 微调模型加载成功:', FINETUNED_MODEL_DIR)
    except Exception as exc:
        print(f'⚠️ 加载微调模型失败，可能是训练输出不完整: {exc}')
else:
    print('⚠️ 未找到微调模型目录:', FINETUNED_MODEL_DIR)

print('\n--- 2. 扫描测试目录 ---')
wav_files = []
for test_dir in TEST_DIRS:
    if test_dir.exists():
        files = sorted(test_dir.rglob('*.wav'))
        wav_files.extend(files)
        print(f'{test_dir}: {len(files)} 个音频文件')
    else:
        print(f'⚠️ 测试目录不存在: {test_dir}')
print(f'共找到 {len(wav_files)} 个音频文件。开始批量测试...')

expected_mapping = load_expected_mapping(SOURCE_JSON_PATH)
print(f'已加载标注数量: {len(expected_mapping)}')

results = []
for wav_path in tqdm(wav_files, desc='推理进度'):
    try:
        file_name = wav_path.name
        rel_path = relative_audio_path(wav_path)
        test_rel_path = file_name
        for test_dir in TEST_DIRS:
            try:
                test_rel_path = wav_path.relative_to(test_dir).as_posix()
                break
            except ValueError:
                pass
        expected = (
            expected_mapping.get(rel_path)
            or expected_mapping.get(test_rel_path)
            or expected_mapping.get(file_name, '')
        )
        base_text = generate_text(base_model, wav_path)
        finetuned_text = generate_text(finetuned_model, wav_path)
        status, base_mark, finetuned_mark = evaluate_status(expected, base_text, finetuned_text)

        results.append({
            'rel_path': rel_path,
            'file': file_name,
            'expected': expected,
            'base': base_text,
            'ft': finetuned_text,
            'status': status,
            'path': str(wav_path),
            'corrected_text': '',
            '_base_mark': base_mark,
            '_ft_mark': finetuned_mark,
        })
    except Exception as exc:
        print(f'文件 {wav_path} 推理出错: {exc}')

print('\n--- 3. 导出 CSV 和 HTML ---')
df = pd.DataFrame(results)
export_columns = ['rel_path', 'file', 'expected', 'base', 'ft', 'status', 'path', 'corrected_text']
if not df.empty:
    df[export_columns].to_csv(CSV_PATH, index=False, encoding='utf-8-sig')
else:
    pd.DataFrame(columns=export_columns).to_csv(CSV_PATH, index=False, encoding='utf-8-sig')

rows_html = []
for i, row in enumerate(results, 1):
    rows_html.append(build_result_row(row, i))

status_counts = df['status'].value_counts().to_dict() if not df.empty else {}
summary_rows_html = []
for status in ['都对', '微调对', '基础对', '都不对', '无标准答案']:
    summary_rows_html.append(f"""
        <tr>
            <td>{status_html(status)}</td>
            <td>{status_counts.get(status, 0)}</td>
        </tr>
    """)

finetune_failures = [
    row for row in results
    if row['status'] in {'基础对', '都不对'} or not row['ft']
]
failure_rows_html = []
for i, row in enumerate(finetune_failures, 1):
    failure_rows_html.append(build_result_row(row, i))

failure_section_html = ''.join(failure_rows_html) if failure_rows_html else """
        <tr>
            <td colspan="6" class="empty">没有需要重点查看的微调失败样本。</td>
        </tr>
"""

html_content = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>ASR 对比测试报告</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 20px; }}
        table {{ border-collapse: collapse; width: 100%; margin-top: 20px; }}
        th, td {{ border: 1px solid #ddd; padding: 10px; text-align: left; vertical-align: top; }}
        th {{ background-color: #f2f2f2; }}
        .diff {{ color: #c00000; font-weight: bold; }}
        .same {{ color: #087a1f; font-weight: bold; }}
        .missing {{ color: #666; }}
        .path {{ font-size: 0.85em; color: #666; word-break: break-all; }}
        .empty {{ color: #666; text-align: center; }}
        .section-title {{ margin-top: 28px; }}
    </style>
</head>
<body>
    <h2>SenseVoice 微调前后对比测试报告</h2>
    <p><b>测试目录:</b> {html.escape(', '.join(str(p) for p in TEST_DIRS))}</p>
    <p><b>微调模型:</b> {html.escape(str(FINETUNED_MODEL_DIR))}</p>
    <p><b>测试文件数:</b> {len(results)}</p>
    <p><b>微调失败/需重点查看:</b> {len(finetune_failures)}</p>
    <p><b>生成时间:</b> {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
    <p><i>状态判定会忽略大小写、空格、标点和特殊符号差异，并与 wav_expected.json 中的 expected_text 比较。</i></p>

    <h3 class="section-title">状态汇总</h3>
    <table>
        <tr>
            <th>状态</th>
            <th>数量</th>
        </tr>
        {''.join(summary_rows_html)}
    </table>

    <h3 class="section-title">微调失败/需重点查看样本</h3>
    <p><i>这里集中列出“基础对但微调错”“基础和微调都错”以及微调输出为空的样本。</i></p>
    <table>
        <tr>
            <th>序号</th>
            <th>相对路径</th>
            <th>原始标注 Expected</th>
            <th>原始基础模型</th>
            <th>微调后模型</th>
            <th>判定结果</th>
        </tr>
        {failure_section_html}
    </table>

    <h3 class="section-title">完整明细</h3>
    <table>
        <tr>
            <th>序号</th>
            <th>相对路径</th>
            <th>原始标注 Expected</th>
            <th>原始基础模型</th>
            <th>微调后模型</th>
            <th>判定结果</th>
        </tr>
        {''.join(rows_html)}
    </table>
</body>
</html>
"""

HTML_PATH.write_text(html_content, encoding='utf-8')

print(f'✅ HTML 报告已生成: {HTML_PATH}')
print(f'✅ CSV 文件已导出至: {CSV_PATH}')
print('\n💡 人工清洗与微调指南')
print("1. 打开 error_analysis.csv。")
print("2. 重点关注 status 为 '都不对' 或 '基础对' 的样本。")
print("3. 在 corrected_text 列填写完全正确的文本；HTML 报告用于浏览检查，CSV 用于后续脚本处理。")

display(HTML(
    f"<b style='color:green;'>对比报告生成成功。</b><br>"
    f"HTML: <code>{html.escape(str(HTML_PATH))}</code><br>"
    f"CSV: <code>{html.escape(str(CSV_PATH))}</code>"
))


## 4. 导出 sensevoice_finetuned_05 为 RKNN

把 `/content/drive/MyDrive/SenseVoice_Project/sensevoice_finetuned_05` 导出为 sherpa-onnx 兼容的 ONNX，再转换为 RK3588 可用的 `model.rknn`。输出目录为 `/content/drive/MyDrive/SenseVoice_Project/sensevoice_finetuned_05_rknn_rk3588`。


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

print('--- 4. 导出 sensevoice_finetuned_05 为 RKNN ---')

PROJECT_DIR = Path('/content/drive/MyDrive/SenseVoice_Project')
FINETUNED_MODEL_DIR = PROJECT_DIR / 'sensevoice_finetuned_05'
ONNX_EXPORT_DIR = PROJECT_DIR / 'sensevoice_finetuned_05_onnx'
RKNN_OUTPUT_DIR = PROJECT_DIR / 'sensevoice_finetuned_05_rknn_rk3588'
TARGET_PLATFORM = 'rk3588'
MAX_LFR_FRAMES = 333  # sherpa-onnx 预训练 RKNN 模型名里的 20 seconds 对应约 333 帧 LFR 特征。

if not FINETUNED_MODEL_DIR.exists():
    raise FileNotFoundError(f'未找到微调模型目录: {FINETUNED_MODEL_DIR}')
if not (FINETUNED_MODEL_DIR / 'model.pt').exists():
    raise FileNotFoundError(f'未找到 model.pt: {FINETUNED_MODEL_DIR / "model.pt"}')

# 1. 安装导出依赖。使用当前 notebook kernel 的 Python，避免 pip 装到别的环境。
# rknn-toolkit2 2.3.x 仍依赖 onnx.mapping；新版 onnx 已移除该属性，所以固定到兼容版本。
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir', '--no-deps',
    'onnx==1.16.1',
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'onnxruntime', 'modelscope', 'sentencepiece', 'soundfile', 'librosa',
], check=True)

# 2. 准备 SenseVoice 源码。sherpa-onnx 的 SenseVoice 导出逻辑依赖 SenseVoice/model.py。
sensevoice_repo = Path('/content/SenseVoice')
if not sensevoice_repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/FunAudioLLM/SenseVoice.git', str(sensevoice_repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(sensevoice_repo / 'requirements.txt')], check=True)

# 3. 准备 FunASR。SenseVoice.from_pretrained 需要 FunASR 组件。
funasr_repo = Path('/content/FunASR')
if not funasr_repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/alibaba-damo-academy/FunASR.git', str(funasr_repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(funasr_repo)], check=True)

for path in (str(sensevoice_repo), str(funasr_repo), str(FINETUNED_MODEL_DIR)):
    if path not in sys.path:
        sys.path.insert(0, path)

import onnx
# 兼容 rknn-toolkit2 对旧 onnx.mapping API 的依赖。
if not hasattr(onnx, 'mapping'):
    from onnx import _mapping
    onnx.mapping = _mapping
import torch
from onnxruntime.quantization import QuantType, quantize_dynamic

# SenseVoice/FunASR import 会加载 torchaudio。Colab 里如果 pip 装到了不匹配的
# torchaudio CUDA wheel，可能报 libcudart.so.xx 找不到。这里强制安装与当前 torch
# 版本/CUDA ABI 匹配的 torchaudio，且 --no-deps 避免把 torch 本身换掉。
def ensure_torchaudio_matches_torch():
    import importlib
    import subprocess
    import sys
    torch_version = torch.__version__.split('+')[0]
    cuda_version = torch.version.cuda
    cuda_tag = 'cu' + cuda_version.replace('.', '') if cuda_version else 'cpu'
    index_url = f'https://download.pytorch.org/whl/{cuda_tag}'

    def purge_torchaudio_modules():
        for name in list(sys.modules):
            if name == 'torchaudio' or name.startswith('torchaudio.'):
                del sys.modules[name]
        importlib.invalidate_caches()

    def install():
        print(f'重装匹配 torch {torch.__version__} 的 torchaudio=={torch_version} ({cuda_tag})...')
        purge_torchaudio_modules()
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'], check=False)
        subprocess.run([
            sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps',
            f'torchaudio=={torch_version}', '--index-url', index_url,
        ], check=True)
        purge_torchaudio_modules()

    try:
        purge_torchaudio_modules()
        import torchaudio  # noqa: F401
        print('torchaudio 可用:', torchaudio.__version__)
    except Exception as exc:
        print('torchaudio 不可用或 CUDA ABI 不匹配:', exc)
        install()
        import torchaudio  # noqa: F401
        print('torchaudio 已修复:', torchaudio.__version__)

ensure_torchaudio_matches_torch()

# Colab 当前 PyTorch 版本在加载 SenseVoice 老代码时，可能遇到部分 nn.Module
# 子类没有完整执行 nn.Module.__init__ 的问题。这里在访问缺失字段时补齐
# state_dict 需要的基础容器，避免 from_pretrained 加载 checkpoint 阶段崩溃。
# 注意：这个补丁是幂等的，避免在同一个 kernel 里重复运行时递归包裹 __getattr__。
import collections
import torch.nn as nn

if not hasattr(nn.Module, '_owon_original_getattr'):
    nn.Module._owon_original_getattr = nn.Module.__getattr__
_orig_module_getattr = nn.Module._owon_original_getattr

_MODULE_DEFAULT_ATTRS = {
    '_parameters': dict,
    '_buffers': dict,
    '_modules': dict,
    '_backward_pre_hooks': collections.OrderedDict,
    '_backward_hooks': collections.OrderedDict,
    '_is_full_backward_hook': lambda: None,
    '_forward_hooks': collections.OrderedDict,
    '_forward_hooks_with_kwargs': collections.OrderedDict,
    '_forward_hooks_always_called': collections.OrderedDict,
    '_forward_pre_hooks': collections.OrderedDict,
    '_forward_pre_hooks_with_kwargs': collections.OrderedDict,
    '_state_dict_hooks': collections.OrderedDict,
    '_state_dict_pre_hooks': collections.OrderedDict,
    '_load_state_dict_pre_hooks': collections.OrderedDict,
    '_load_state_dict_post_hooks': collections.OrderedDict,
    '_non_persistent_buffers_set': set,
}

def _ensure_module_internals(module):
    for attr, factory in _MODULE_DEFAULT_ATTRS.items():
        if attr not in module.__dict__:
            object.__setattr__(module, attr, factory())

def _module_getattr_compat(self, name):
    if name in _MODULE_DEFAULT_ATTRS:
        _ensure_module_internals(self)
        return self.__dict__[name]
    return _orig_module_getattr(self, name)

nn.Module.__getattr__ = _module_getattr_compat

from model import SenseVoiceSmall


def safe_rmtree(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)


def add_meta_data(filename, meta_data):
    model = onnx.load(filename)
    while len(model.metadata_props):
        model.metadata_props.pop()
    for key, value in meta_data.items():
        meta = model.metadata_props.add()
        meta.key = key
        meta.value = str(value)
    onnx.save(model, filename)


def force_static_onnx_inputs(filename):
    model = onnx.load(filename)
    static_shapes = {
        'x': [1, MAX_LFR_FRAMES, 560],
    }
    for value_info in model.graph.input:
        shape = static_shapes.get(value_info.name)
        if not shape:
            continue
        dims = value_info.type.tensor_type.shape.dim
        while len(dims) < len(shape):
            dims.add()
        for dim, value in zip(dims, shape):
            dim.ClearField('dim_param')
            dim.dim_value = int(value)
    onnx.save(model, filename)
    reloaded = onnx.load(filename)
    print('ONNX 输入 shape:')
    for value_info in reloaded.graph.input:
        dims = []
        for dim in value_info.type.tensor_type.shape.dim:
            dims.append(dim.dim_value if dim.HasField('dim_value') else dim.dim_param)
        print(' ', value_info.name, dims)


def load_cmvn(filename):
    neg_mean = None
    inv_stddev = None
    with open(filename, encoding='utf-8') as f:
        for line in f:
            if not line.startswith('<'):
                continue
            values = line.split()[3:-1]
            if neg_mean is None:
                neg_mean = ','.join(values)
            else:
                inv_stddev = ','.join(values)
    return neg_mean, inv_stddev


def generate_tokens(params, tokens_path):
    sp = params['tokenizer'].sp
    with open(tokens_path, 'w', encoding='utf-8') as f:
        for i in range(sp.vocab_size()):
            f.write(f'{sp.id_to_piece(i)} {i}\n')


def modified_forward(self, x):
    # RKNN 不支持 int32 graph input。导出时固定中文/不逆文本归一化，并只保留声学特征 x 作为输入。
    x_length = torch.tensor([MAX_LFR_FRAMES], dtype=torch.int32, device=x.device)
    language = torch.tensor([EXPORT_LANGUAGE_ID], dtype=torch.long, device=x.device)
    text_norm = torch.tensor([EXPORT_TEXT_NORM_ID], dtype=torch.long, device=x.device)
    language_query = self.embed(language).unsqueeze(1)
    text_norm_query = self.embed(text_norm).unsqueeze(1)
    event_emo_query = self.embed(torch.LongTensor([[1, 2]]).to(x.device)).repeat(x.size(0), 1, 1)
    x = torch.cat((language_query, event_emo_query, text_norm_query, x), dim=1)
    x_length += 4
    encoder_out, encoder_out_lens = self.encoder(x, x_length)
    if isinstance(encoder_out, tuple):
        encoder_out = encoder_out[0]
    ctc_logits = self.ctc.ctc_lo(encoder_out)
    return ctc_logits


print('清理并创建 ONNX/RKNN 输出目录...')
safe_rmtree(ONNX_EXPORT_DIR)
safe_rmtree(RKNN_OUTPUT_DIR)
ONNX_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
RKNN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('加载微调模型:', FINETUNED_MODEL_DIR)
model, params = SenseVoiceSmall.from_pretrained(model=str(FINETUNED_MODEL_DIR), device='cpu')
model.eval()
EXPORT_LANGUAGE_ID = int(model.lid_dict['zh'])
EXPORT_TEXT_NORM_ID = int(model.textnorm_dict['woitn'])
model.__class__.forward = modified_forward

onnx_path = ONNX_EXPORT_DIR / 'model.onnx'
int8_onnx_path = ONNX_EXPORT_DIR / 'model.int8.onnx'
tokens_path = ONNX_EXPORT_DIR / 'tokens.txt'

print('生成 tokens.txt...')
generate_tokens(params, tokens_path)

print('导出 ONNX:', onnx_path)
x = torch.randn(1, MAX_LFR_FRAMES, 560, dtype=torch.float32)

with torch.no_grad():
    torch.onnx.export(
        model,
        (x,),
        str(onnx_path),
        opset_version=13,
        input_names=['x'],
        output_names=['logits'],
    )

lfr_window_size = params['frontend_conf']['lfr_m']
lfr_window_shift = params['frontend_conf']['lfr_n']
neg_mean, inv_stddev = load_cmvn(params['frontend_conf']['cmvn_file'])
vocab_size = params['tokenizer'].sp.vocab_size()
meta_data = {
    'lfr_window_size': lfr_window_size,
    'lfr_window_shift': lfr_window_shift,
    'normalize_samples': 0,
    'neg_mean': neg_mean,
    'inv_stddev': inv_stddev,
    'model_type': 'sense_voice_ctc',
    'version': '2',
    'model_author': 'iic',
    'maintainer': 'owon/k2-fsa',
    'vocab_size': vocab_size,
    'comment': str(FINETUNED_MODEL_DIR),
    'lang_auto': model.lid_dict['auto'],
    'lang_zh': model.lid_dict['zh'],
    'lang_en': model.lid_dict['en'],
    'lang_yue': model.lid_dict['yue'],
    'lang_ja': model.lid_dict['ja'],
    'lang_ko': model.lid_dict['ko'],
    'lang_nospeech': model.lid_dict['nospeech'],
    'with_itn': model.textnorm_dict['withitn'],
    'without_itn': model.textnorm_dict['woitn'],
    'url': 'https://github.com/FunAudioLLM/SenseVoice',
}
add_meta_data(str(onnx_path), meta_data)
force_static_onnx_inputs(str(onnx_path))

print('生成动态量化 ONNX:', int8_onnx_path)
quantize_dynamic(
    model_input=str(onnx_path),
    model_output=str(int8_onnx_path),
    op_types_to_quantize=['MatMul'],
    weight_type=QuantType.QUInt8,
)

print('安装/导入 rknn-toolkit2...')

# rknn-toolkit2 初始化时会通过 pkg_resources 检查 opencv-python。
# Colab 多次 pip 安装后偶尔会留下缺 METADATA 的 opencv_python-*.dist-info，
# 导致 RKNN(verbose=True) 阶段 FileNotFoundError。这里先强制修复 OpenCV 元数据。
def repair_opencv_metadata():
    import importlib
    import subprocess
    import sys
    try:
        import pkg_resources
        pkg_resources.get_distribution('opencv-python').requires()
        import cv2  # noqa: F401
        print('OpenCV 元数据可用:', cv2.__version__)
        return
    except Exception as exc:
        print('OpenCV 元数据不可用，正在重装 opencv-python:', exc)

    for name in list(sys.modules):
        if name == 'cv2' or name.startswith('cv2.'):
            del sys.modules[name]
    importlib.invalidate_caches()
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'opencv-python', 'opencv-python-headless', 'opencv-contrib-python'], check=False)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps',
        'opencv-python==4.10.0.84',
    ], check=True)
    importlib.invalidate_caches()
    import pkg_resources
    pkg_resources.get_distribution('opencv-python').requires()
    import cv2  # noqa: F401
    print('OpenCV 已修复:', cv2.__version__)

repair_opencv_metadata()

# rknn-toolkit2 的 wheel 有时不会完整拉起运行时依赖；显式补齐常见依赖。
def install_rknn_runtime_deps():
    import importlib
    import importlib.metadata as importlib_metadata
    import site
    import subprocess
    import sys

    deps = [
        'ruamel.yaml==0.18.6',
        'ruamel.yaml.clib',
        'protobuf',
        'psutil',
    ]
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps',
        *deps,
    ], check=True)

    importlib.invalidate_caches()
    import pkg_resources
    importlib.reload(pkg_resources)

    # 重建 working_set，而不是只在旧缓存上 add_entry。
    pkg_resources.working_set = pkg_resources.WorkingSet(sys.path)
    for site_dir in site.getsitepackages() + [site.getusersitepackages()]:
        if site_dir:
            pkg_resources.working_set.add_entry(site_dir)
    for path_entry in sys.path:
        if path_entry:
            pkg_resources.working_set.add_entry(path_entry)

    try:
        dist = pkg_resources.get_distribution('ruamel.yaml')
        print('ruamel.yaml 可见:', dist.version, dist.location)
        return
    except Exception as exc:
        print('pkg_resources 仍未找到 ruamel.yaml:', exc)
        subprocess.run([sys.executable, '-m', 'pip', 'show', 'ruamel.yaml'], check=False)
        print('importlib.metadata 中的 ruamel 发行包:')
        for dist in importlib_metadata.distributions():
            name = dist.metadata.get('Name', '')
            if 'ruamel' in name.lower():
                print('  ', name, dist.version, dist.locate_file(''))
        raise

install_rknn_runtime_deps()

try:
    from rknn.api import RKNN
except ModuleNotFoundError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rknn-toolkit2'], check=True)
    from rknn.api import RKNN

def ensure_onnx_mapping_for_rknn():
    import types
    import numpy as np
    import onnx
    from onnx import TensorProto

    # RKNN 2.3.x 读取旧版 onnx.mapping，并且期望 value 是 np.dtype，
    # 不是 np.float32 这类 class。Colab 中多次运行后也要强制覆盖成兼容格式。
    tensor_type_to_np_type = {
        TensorProto.FLOAT: np.dtype('float32'),
        TensorProto.UINT8: np.dtype('uint8'),
        TensorProto.INT8: np.dtype('int8'),
        TensorProto.UINT16: np.dtype('uint16'),
        TensorProto.INT16: np.dtype('int16'),
        TensorProto.INT32: np.dtype('int32'),
        TensorProto.INT64: np.dtype('int64'),
        TensorProto.BOOL: np.dtype('bool'),
        TensorProto.FLOAT16: np.dtype('float16'),
        TensorProto.DOUBLE: np.dtype('float64'),
        TensorProto.UINT32: np.dtype('uint32'),
        TensorProto.UINT64: np.dtype('uint64'),
        TensorProto.COMPLEX64: np.dtype('complex64'),
        TensorProto.COMPLEX128: np.dtype('complex128'),
    }
    np_type_to_tensor_type = {v: k for k, v in tensor_type_to_np_type.items()}

    onnx.mapping = types.SimpleNamespace(
        TENSOR_TYPE_TO_NP_TYPE=tensor_type_to_np_type,
        NP_TYPE_TO_TENSOR_TYPE=np_type_to_tensor_type,
    )
    print('onnx.mapping 已规范化为 RKNN 兼容 dtype')

ensure_onnx_mapping_for_rknn()

rknn_path = RKNN_OUTPUT_DIR / 'model.rknn'
print('转换 RKNN:', rknn_path)
rknn = RKNN(verbose=True)
rknn.config(target_platform=TARGET_PLATFORM)
ret = rknn.load_onnx(model=str(onnx_path))
if ret != 0:
    raise RuntimeError(f'rknn.load_onnx 失败: {ret}')

# 不做 RKNN PTQ 校准，先保证能生成可部署模型；如需更小/更快模型，再补 feature 校准集做量化。
ret = rknn.build(do_quantization=False)
if ret != 0:
    raise RuntimeError(f'rknn.build 失败: {ret}')
ret = rknn.export_rknn(str(rknn_path))
if ret != 0:
    raise RuntimeError(f'rknn.export_rknn 失败: {ret}')
rknn.release()

shutil.copy2(tokens_path, RKNN_OUTPUT_DIR / 'tokens.txt')

print('\n✅ RKNN 转换完成')
print('ONNX 目录:', ONNX_EXPORT_DIR)
print('RKNN 目录:', RKNN_OUTPUT_DIR)
print('文件列表:')
for item in sorted(RKNN_OUTPUT_DIR.iterdir()):
    print(item.name, item.stat().st_size)
